# Day 29: Prediction API

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Expose the existing validated model through local health and prediction endpoints.

## Theory

An API separates the transport from model inference. The request handler calls the same prediction function as other interfaces, and never retrains the model or accepts model uploads.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Call both endpoints in-process using a real reference record.

In [2]:
from fastapi.testclient import TestClient
from api import app
from src.dataset import original_split
client=TestClient(app)
response=client.get("/health")
assert response.status_code==200
print(response.json())
record=original_split()[1].head(1).to_dict("records")
r=client.post("/predict",json={"records":record})
assert r.status_code==200
print(json.dumps(r.json(),indent=2))

C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


{'status': 'ready', 'model_id': '1bedf35d0d56c4f0'}
{
  "model_id": "1bedf35d0d56c4f0",
  "predictions": [
    {
      "predicted_category": "Medium",
      "model_estimated_proba_High": 0.0,
      "model_estimated_proba_Low": 0.09090909090909091,
      "model_estimated_proba_Medium": 0.9090909090909091
    }
  ]
}


Verify malformed payloads are rejected.

In [3]:
bad=client.post("/predict",json={"records":[]})
assert bad.status_code==422
assert client.post("/predict",json={"records":[dict(record[0],G3=20)]}).status_code==422
print("Empty and target-leaking payload rejection: PASS")

Empty and target-leaking payload rejection: PASS


## Findings and interpretation

The endpoints return the verified artifact identity and actual prediction probabilities. Validation errors use 422 and model availability errors use 503.

## Limitations

The TestClient check exercises the ASGI application in-process. A live HTTP startup check is scheduled for deployment verification. This API is intended for local demonstration.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Expose the existing validated model through local health and prediction endpoints.

## Next-day plan

Expand API tests and document the request/response contract.